# HW04

姓名：邓烨涛  
学号：20234080108





In [1]:
import math
import re
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


## 2 序列模型

### 2.1 理论题

给定序列 `"ababc"`，用二元语言模型估计 $p(x_t|x_{t-1})$。序列中的相邻字符对为：

$$
(a,b),\ (b,a),\ (a,b),\ (b,c).
$$

当上一个字符为 `b` 时，一共出现了 2 次转移：

$$
C(b,a)=1,\quad C(b,c)=1,\quad C(b,b)=0.
$$

词表为 `{'a','b','c'}`，大小 $|V|=3$。使用加 1 平滑：

$$
p(x|b)=\frac{C(b,x)+1}{C(b,\cdot)+|V|}.
$$

其中 $C(b,\cdot)=2$，所以分母为 $2+3=5$。

因此：

$$
p('a'|'b')=\frac{1+1}{5}=\frac25=0.4.
$$

$$
p('c'|'b')=\frac{1+1}{5}=\frac25=0.4.
$$

加 1 平滑会给没有出现过的转移也分配一个非零概率，例如 $p('b'|'b')=\frac15$。


### 2.2 编程题：文本预处理与 n-gram 样本

函数 `preprocess_text(text, n)` 完成以下步骤：

1. 将文本转为小写。
2. 按单词切分。
3. 为每个词分配整数 ID，从 0 开始。
4. 生成用于语言模型的 n-gram 特征序列和标签序列。

这里采用的样本格式是：每个特征由连续的 `n` 个词组成，标签为紧随其后的下一个词；如果已经到达序列末尾，则标签为 `None`。


In [2]:
def preprocess_text(text, n):
    text = text.lower()
    tokens = re.findall(r"[a-z0-9']+", text)

    vocab = {}
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

    token_ids = [vocab[token] for token in tokens]
    features, labels = [], []
    for i in range(max(0, len(tokens) - n + 1)):
        features.append(tokens[i:i + n])
        next_pos = i + n
        labels.append(tokens[next_pos] if next_pos < len(tokens) else None)

    return {
        "tokens": tokens,
        "vocab": vocab,
        "token_ids": token_ids,
        "features": features,
        "labels": labels,
    }


result = preprocess_text("The time machine", n=2)
print(result["tokens"])
print(result["vocab"])
print(result["token_ids"])
print(result["features"])
print(result["labels"])


['the', 'time', 'machine']
{'the': 0, 'time': 1, 'machine': 2}
[0, 1, 2]
[['the', 'time'], ['time', 'machine']]
['machine', None]


## 3 循环神经网络

### 3.1 理论题：线性 RNN 中 $\partial L/\partial W_{hh}$

线性 RNN 定义为：

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t,
$$

$$
o_t=W_{oh}h_t.
$$

损失函数为：

$$
L=\frac12\sum_{t=1}^{T}(o_t-y_t)^2.
$$

为了求 $W_{hh}$ 的梯度，需要使用时间反向传播（BPTT）。令

$$
e_t=\frac{\partial L}{\partial o_t}=o_t-y_t.
$$

又因为 $o_t=W_{oh}h_t$，所以当前时刻输出对隐藏状态的梯度贡献为

$$
W_{oh}^T e_t.
$$

但 $h_t$ 还会影响未来的 $h_{t+1},h_{t+2},\ldots$，因此设

$$
\delta_t=\frac{\partial L}{\partial h_t}.
$$

递推式为：

$$
\delta_t=W_{oh}^T(o_t-y_t)+W_{hh}^T\delta_{t+1},
$$

其中 $\delta_{T+1}=0$。

由于

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t,
$$

所以单个时刻对 $W_{hh}$ 的梯度为

$$
\frac{\partial L}{\partial W_{hh}}\bigg|_t=\delta_t h_{t-1}^T.
$$

最终：

$$
\frac{\partial L}{\partial W_{hh}}=\sum_{t=1}^{T}\delta_t h_{t-1}^T.
$$

这说明 RNN 的参数在所有时间步共享，所以梯度需要把所有时间步的贡献累加起来。


### 3.2 编程题：手写 RNN Cell 反向传播

使用 `tanh` 激活：

$$
a_t=x_tW_{hx}+h_{t-1}W_{hh}+b_h,
$$

$$
h_t=\tanh(a_t).
$$

给定上游梯度 `dh_next`，下面计算 `dx_t, dh_prev, dW_hx, dW_hh, db_h`。


In [3]:
def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = torch.tanh(a_t)
    cache = (x_t, h_prev, W_hx, W_hh, a_t, h_t)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, a_t, h_t = cache
    da = dh_next * (1 - h_t ** 2)
    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = da.sum(dim=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


batch_size, input_size, hidden_size = 2, 3, 4
x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
W_hx = torch.randn(input_size, hidden_size)
W_hh = torch.randn(hidden_size, hidden_size)
b_h = torch.randn(hidden_size)
dh_next = torch.randn(batch_size, hidden_size)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
grads = rnn_cell_backward(dh_next, cache)
for name, value in zip(["dx_t", "dh_prev", "dW_hx", "dW_hh", "db_h"], grads):
    print(name, tuple(value.shape))


dx_t (2, 3)
dh_prev (2, 4)
dW_hx (3, 4)
dW_hh (4, 4)
db_h (4,)


## 4 双向 RNN

### 4.1 理论题：参数量

设双向 RNN 有 $L$ 层，每层每个方向的隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$。这里按普通 RNN 计算，每个方向每层有输入到隐藏矩阵、隐藏到隐藏矩阵和偏置项。

第一层每个方向的参数量为：

$$
DH+H^2+H.
$$

因为是双向，所以第一层为：

$$
2(DH+H^2+H).
$$

从第二层开始，每层输入来自上一层的正向和反向隐藏状态拼接，因此输入维度为 $2H$。每个方向参数量为：

$$
2H\cdot H+H^2+H=3H^2+H.
$$

双向后每层为：

$$
2(3H^2+H)=6H^2+2H.
$$

共有 $L-1$ 个这样的高层。最后如果接一个线性输出层，从 $2H$ 映射到 $O$，参数量为：

$$
2HO+O.
$$

所以总参数量为：

$$
2(DH+H^2+H)+(L-1)(6H^2+2H)+2HO+O.
$$


### 4.2 编程题：使用 `torch.nn.RNN` 实现双向 RNN

输入 $X$ 的形状为 `(seq_len, batch, input_dim)`。双向 RNN 的每个时间步输出会拼接正向和反向隐藏状态，所以输出形状为 `(seq_len, batch, 2*hidden_dim)`。


In [4]:
class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
        )

    def forward(self, X):
        outputs, h_n = self.rnn(X)
        # 取最后一层的正向和反向最终状态作为整个序列的表示。
        forward_last = h_n[-2]
        backward_last = h_n[-1]
        sequence_repr = torch.cat([forward_last, backward_last], dim=-1)
        return outputs, sequence_repr


seq_len, batch, input_dim, hidden_dim = 5, 3, 4, 6
encoder = BiRNNEncoder(input_dim, hidden_dim)
X = torch.randn(seq_len, batch, input_dim)
outputs, sequence_repr = encoder(X)
print("outputs:", outputs.shape)
print("sequence_repr:", sequence_repr.shape)


outputs: torch.Size([5, 3, 12])
sequence_repr: torch.Size([3, 12])


## 5 嵌入向量

### 5.1 理论题：Skip-gram 负采样损失

在 Skip-gram 中，中心词为 $w_c$，上下文正样本词为 $w_o$。设中心词向量为 $v_c$，正样本输出词向量为 $u_o$，负样本词向量为 $u_{k}$，一共采样 $K$ 个负样本。

负采样的目标是让正样本内积更大，让负样本内积更小。单个中心词和上下文词对的损失为：

$$
\ell=-\log\sigma(u_o^T v_c)-\sum_{k=1}^{K}\log\sigma(-u_k^T v_c).
$$

其中 $\sigma(x)=1/(1+e^{-x})$。第一项鼓励模型把真实上下文词预测为正例；第二项鼓励模型把噪声词预测为负例。


### 5.2 编程题：CBOW 完整 Softmax 损失

CBOW 使用上下文词预测中心词。这里输入为上下文词 ID，先查表得到词向量，再对上下文向量取平均，最后经过输出矩阵得到词表大小的 logits，并用交叉熵计算损失。


In [5]:
def cbow_full_softmax_loss(context_ids, target_ids, W, W_out):
    # context_ids: (batch, context_size)
    # W: (V, d), W_out: (d, V)
    context_embeds = W[context_ids]
    hidden = context_embeds.mean(dim=1)
    logits = hidden @ W_out
    loss = F.cross_entropy(logits, target_ids)
    return loss, logits


V, d, batch, context_size = 8, 5, 3, 4
W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)
context_ids = torch.tensor([[0, 1, 2, 3], [2, 3, 4, 5], [1, 3, 5, 7]])
target_ids = torch.tensor([4, 1, 6])

loss, logits = cbow_full_softmax_loss(context_ids, target_ids, W, W_out)
loss.backward()
print("logits shape:", logits.shape)
print("loss:", loss.item())
print("W.grad shape:", W.grad.shape)
print("W_out.grad shape:", W_out.grad.shape)


logits shape: torch.Size([3, 8])
loss: 1.880052089691162
W.grad shape: torch.Size([8, 5])
W_out.grad shape: torch.Size([5, 8])


## 6 注意力机制

### 6.1 理论题：缩放点积注意力的矩阵形状

题目给出：

$$
Q\in \mathbb{R}^{2\times 4},\quad K\in \mathbb{R}^{3\times 4},\quad V\in \mathbb{R}^{3\times 5}.
$$

缩放点积注意力的 score 为：

$$
\text{score}=\frac{QK^T}{\sqrt{d_k}},
$$

其中 $d_k=4$，所以分母为 $\sqrt4=2$。

矩阵形状为：

$$
QK^T:(2\times 4)(4\times 3)=2\times 3.
$$

对 score 的每一行做 softmax，得到注意力权重矩阵：

$$
A=\text{softmax}\left(\frac{QK^T}{2}\right)\in\mathbb{R}^{2\times 3}.
$$

最后输出：

$$
AV:(2\times 3)(3\times 5)=2\times 5.
$$

因此该注意力层的输出矩阵形状是 $2\times 5$。


In [6]:
Q = torch.randn(2, 4)
K = torch.randn(3, 4)
V_mat = torch.randn(3, 5)

scores = Q @ K.T / math.sqrt(4)
attn = torch.softmax(scores, dim=-1)
out = attn @ V_mat

print("scores shape:", scores.shape)
print("attention shape:", attn.shape)
print("output shape:", out.shape)
print("row sums:", attn.sum(dim=-1))


scores shape: torch.Size([2, 3])
attention shape: torch.Size([2, 3])
output shape: torch.Size([2, 5])
row sums: tensor([1., 1.])


### 6.2 编程题：实现 Multi-Head Attention

设 `num_heads=2`，`d_model=4`，则每个头的维度为：

$$
d_k=d_v=d_{model}/num\_heads=2.
$$

输入 $X$ 的形状为 `(seq_len, batch, d_model)`，输出形状保持不变。


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_heads(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        X = X.view(seq_len, batch, self.num_heads, self.head_dim)
        return X.permute(1, 2, 0, 3)  # (batch, heads, seq_len, head_dim)

    def _combine_heads(self, X):
        # X: (batch, heads, seq_len, head_dim)
        batch, heads, seq_len, head_dim = X.shape
        X = X.permute(2, 0, 1, 3).contiguous()
        return X.view(seq_len, batch, heads * head_dim)

    def forward(self, X):
        Q = self._split_heads(self.W_q(X))
        K = self._split_heads(self.W_k(X))
        V = self._split_heads(self.W_v(X))

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        context = attn @ V
        combined = self._combine_heads(context)
        return self.W_o(combined)


seq_len, batch, d_model = 6, 2, 4
mha = MultiHeadSelfAttention(d_model=4, num_heads=2)
X = torch.randn(seq_len, batch, d_model)
Y = mha(X)
print("input shape:", X.shape)
print("output shape:", Y.shape)


input shape: torch.Size([6, 2, 4])
output shape: torch.Size([6, 2, 4])


: 

小结：本次作业主要围绕序列建模展开。n-gram 模型用条件概率描述局部词序关系；RNN 通过隐藏状态传递历史信息；双向 RNN 同时利用前后文；词嵌入把离散词映射到连续向量空间；注意力机制则通过 query、key、value 的匹配关系动态聚合信息。
